# Customer Segmentation Using Clustering - Review 2

**Scope of this review**
1. Data preprocessing
2. K-Means clustering
3. Hierarchical clustering (Agglomerative, Ward linkage)
4. DBSCAN

**Dataset:** `Customer_Behaviour_Data.csv` (1,000 customers, 8 features)

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

sns.set_style("whitegrid")

## 2. Load the Dataset
Run the cell and click **Choose Files** to upload `Customer_Behaviour_Data.csv`.

In [ ]:
csv_name = "Customer_Behaviour_Data.csv"
try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        csv_name = list(uploaded.keys())[0]
        print("Uploaded:", csv_name)
except ImportError:
    print("Not running in Colab - looking for", csv_name, "in the current folder")

df = pd.read_csv(csv_name)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()
print("\nMissing values:\n", df.isnull().sum())
df.describe().round(1)

In [ ]:
features = ["Age", "Annual_Income_k", "Spending_Score", "Purchase_Frequency",
            "Avg_Order_Value", "Total_Purchases", "Website_Visits", "Days_Since_Last_Purchase"]

df[features].hist(figsize=(15, 8), bins=30)
plt.tight_layout()
plt.show()

# PART A: Data Preprocessing

### A1. Remove duplicates and fix data types

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

for col in features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Gender"] = df["Gender"].astype(str).str.strip().str.title().replace("Nan", np.nan)
print("Shape after removing duplicates:", df.shape)

### A2. Handle missing values
Numeric columns are filled with the **median**. `Gender` is filled with the **mode**.

In [ ]:
print("Missing before:", int(df.isnull().sum().sum()))
for col in features:
    df[col] = df[col].fillna(df[col].median())
df["Gender"] = df["Gender"].fillna(df["Gender"].mode()[0])
print("Missing after:", int(df.isnull().sum().sum()))

X = df[features].to_numpy()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[features].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation between features")
plt.show()

# PART B: Clustering Algorithms

We apply the same preprocessed data `X` to all three algorithms and compare them.

## B1. K-Means

**How it works:** choose k, place k centroids, assign every customer to the nearest centroid, move each centroid to the mean of its customers, repeat until nothing changes.

**Choosing k:** the elbow method (inertia) and the silhouette score.

In [ ]:
k_range = range(2, 11)
inertias, sil_scores = [], []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(k_range, inertias, "o-"); ax[0].set(title="Elbow method", xlabel="k", ylabel="Inertia")
ax[1].plot(k_range, sil_scores, "o-", color="green"); ax[1].set(title="Silhouette score", xlabel="k")
plt.tight_layout(); plt.show()

best_k = list(k_range)[int(np.argmax(sil_scores))]
print(f"Highest silhouette score: k = {best_k} ({max(sil_scores):.3f})")

The silhouette score is highest at k = 3, but 3 groups are too coarse for marketing. We use **k = 5**, which still has a good score and gives clearly different customer types.

In [ ]:
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
df["KMeans"] = kmeans.fit_predict(X)

print("K-Means cluster sizes:")
print(df["KMeans"].value_counts().sort_index())

## B2. Hierarchical Clustering (Agglomerative, Ward linkage)

**How it works:** every customer starts as its own cluster. The two closest clusters are merged repeatedly (Ward linkage merges the pair that increases variance the least) until k clusters remain. The result can be drawn as a **dendrogram**.

In [ ]:
Z = linkage(X, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=9, show_contracted=True)
plt.title("Dendrogram (Ward linkage, last 30 merges)")
plt.xlabel("Cluster size / sample"); plt.ylabel("Distance")
plt.tight_layout(); plt.show()

In [ ]:
agg = AgglomerativeClustering(n_clusters=n_clusters, linkage="ward")
df["Hierarchical"] = agg.fit_predict(X)

print("Hierarchical cluster sizes:")
print(df["Hierarchical"].value_counts().sort_index())

## B3. DBSCAN

**How it works:** a point is a *core point* if at least `min_samples` points lie within distance `eps`. Core points that are close together form a cluster. Points that do not belong to any cluster are labelled **noise (-1)**. DBSCAN does not need k.

**Choosing eps:** plot each point's distance to its 10th nearest neighbour, sorted. The "knee" of the curve is a good eps.

In [ ]:
min_samples = 10
nn = NearestNeighbors(n_neighbors=min_samples).fit(X)
distances, _ = nn.kneighbors(X)
k_dist = np.sort(distances[:, -1])

plt.figure(figsize=(8, 4))
plt.plot(k_dist)
plt.axhline(1.3, color="red", linestyle="--", label="eps = 1.3")
plt.title("k-distance plot (k = 10)")
plt.xlabel("Points sorted by distance"); plt.ylabel("Distance to 10th neighbour")
plt.legend(); plt.show()

In [ ]:
dbscan = DBSCAN(eps=1.3, min_samples=min_samples)
df["DBSCAN"] = dbscan.fit_predict(X)

print("DBSCAN cluster sizes (-1 = noise):")
print(df["DBSCAN"].value_counts().sort_index())

# PART C: Comparison of the Three Algorithms

In [ ]:
def evaluate(name, labels):
    mask = labels != -1                       # ignore DBSCAN noise points
    n = len(set(labels[mask]))
    return {"Algorithm": name,
            "Clusters": n,
            "Noise points": int((~mask).sum()),
            "Silhouette (higher is better)": round(silhouette_score(X[mask], labels[mask]), 3),
            "Davies-Bouldin (lower is better)": round(davies_bouldin_score(X[mask], labels[mask]), 3)}

results = pd.DataFrame([evaluate("K-Means", df["KMeans"].values),
                        evaluate("Hierarchical (Ward)", df["Hierarchical"].values),
                        evaluate("DBSCAN", df["DBSCAN"].values)])
results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, m in zip(axes, ["KMeans", "Hierarchical", "DBSCAN"]):
    sns.scatterplot(data=df, x="Annual_Income_k", y="Spending_Score", hue=m,
                    palette="tab10", s=35, ax=ax)
    ax.set_title(m)
plt.tight_layout(); plt.show()

In [ ]:
# How closely do K-Means and Hierarchical agree? (1.0 = identical grouping)
from sklearn.metrics import adjusted_rand_score
print("Agreement between K-Means and Hierarchical (ARI):",
      round(adjusted_rand_score(df["KMeans"], df["Hierarchical"]), 3))

## Observations

- **K-Means** and **Hierarchical (Ward)** give almost the same scores and groupings, which shows the five segments are stable and not an accident of one algorithm.
- **DBSCAN** finds fewer clusters and labels some customers as noise, because the customer groups touch each other without clear low-density gaps. Its silhouette score is calculated without the noise points, so it is not a fair direct comparison.
- K-Means is the best choice for this data: fast, easy to interpret, and it gives a fixed number of segments.